# Avaliação local de candidatos TTS para português brasileiro

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/BloomRX/Lia-Code/blob/arena/01a0ec89-lia-code/benchmark/colab_ptbr_tts_eval.ipynb)

Este notebook compara inferência com clonagem de voz no **Chatterbox PT-BR** e no **Qwen3-TTS Base 0.6B**, e termina com um A/B controlado de expressividade. Não treina modelos. O WAV permanece no runtime privado do Colab; a transcrição auxiliar é feita localmente com Whisper-small, mostrada para revisão e não enviada a serviços externos. O ASR serve somente para preparar `ref_text` neste benchmark, não é integrado à Lia.

## Antes de executar
1. No Colab, selecione **Runtime → Change runtime type → T4 GPU** (ou outra GPU CUDA disponível).
2. Execute a célula abaixo e carregue um WAV original limpo da Lia quando solicitado.
3. Escute as amostras e compare-as; o português anunciado pelo modelo não é, por si só, prova de qualidade regional PT-BR.

A célula instala Torch/Torchaudio 2.8.0 CUDA 12.8 em `/content/lia_tts_candidate_eval/torch_runtime`, conforme as versões declaradas pelos Spaces, e Protobuf compatível no mesmo caminho. Esses pacotes ficam isolados do runtime global. Como TTS não usa visão, a integração opcional com a `torchvision` global do Colab é desativada nos processos de inferência. Na T4, Qwen é testado em FP32: a equipe Qwen recomenda FP32 em GPUs sem suporte BF16 após registrar instabilidade de amostragem FP16 no code predictor ([issue upstream](https://github.com/QwenLM/Qwen3-TTS/issues/43)). A pré-verificação imprime os caminhos/versões usados; os logs e tracebacks dos processos aparecem diretamente na célula. As amostras concluídas do Chatterbox são reutilizadas em execuções posteriores. WAV, modelos e saídas ficam em `/content/lia_tts_candidate_eval`.

Versões consultadas: [requirements do Space Chatterbox PT-BR](https://huggingface.co/spaces/ResembleAI/Chatterbox-Multilingual-TTS-pt-br/resolve/9e515821e826e207cd617a0fdd0223899ed108ea/requirements.txt), [requirements do Space Qwen3-TTS](https://huggingface.co/spaces/Qwen/Qwen3-TTS/resolve/main/requirements.txt) e [índice oficial PyTorch CUDA 12.8](https://download.pytorch.org/whl/cu128/). Python 3.13 possui wheels CPython para essa pilha. Se o runtime for reiniciado, será necessário carregar o WAV novamente. O notebook transcreve automaticamente o WAV completo com Whisper-small (modelo Apache-2.0, revisão fixada), limita a saída em português, exibe a hipótese e aceita Enter para usá-la ou texto corrigido. Só então envia esse `ref_text` local ao Qwen; nenhum áudio sai do Colab.

In [ ]:
# Cole este arquivo inteiro em UMA célula do Colab e execute.
# O teste é de inferência/clonagem, NÃO inicia fine-tuning.
# A gravação de referência fica no runtime Colab; não é enviada aos Spaces/demos.
# Modelos/pacotes ficam em /content/lia_tts_candidate_eval (fora do Git).

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import textwrap
import threading
import time
import venv

# ---------- Configuração que pode ser alterada sem editar o restante ----------
ROOT = Path('/content/lia_tts_candidate_eval')
REF_WAV = ROOT / 'lia_original.wav'
# A transcrição do WAV inteiro é gerada localmente e mostrada para revisão antes do Qwen.
REF_TEXT = ''
TEXTS = [
    'Não é por vocês serem velhos e acabados.',
    'O ônibus chegou às seis; vou passar no mercado e comprar pão de queijo.',
    'Você está falando sério? A gente se vê depois, tá bom?',
]

# Revisões fixas para que o teste seja repetível.
SPACE_ID = 'ResembleAI/Chatterbox-Multilingual-TTS-pt-br'
SPACE_REV = '9e515821e826e207cd617a0fdd0223899ed108ea'
CHATTERBOX_PTBR_ID = 'ResembleAI/Chatterbox-Multilingual-pt-br'
CHATTERBOX_PTBR_REV = 'b3952f18bc2eaa72b9bd7c17d2c4653bcad4770d'
CHATTERBOX_BASE_ID = 'ResembleAI/chatterbox'
CHATTERBOX_BASE_REV = '5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18'
QWEN_ID = 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
QWEN_REV = '5d83992436eae1d760afd27aff78a71d676296fc'
# O limite curto é suficiente para as três frases de teste e reduz o impacto de EOS ausente.
QWEN_LANGUAGE = 'Portuguese'  # idioma explicitamente passado à API; não usar detecção automática
QWEN_MAX_NEW_TOKENS = 128
QWEN_MAX_AUDIO_SECONDS = 15.0
# Frase afetivamente marcada para uma comparação de prosódia, não de persona.
EXPRESSION_TEST_TEXT = 'Você conseguiu! Eu sabia que era capaz. Estou muito orgulhosa de você!'
# ASR exclusivamente auxiliar à avaliação; whisper-small é Apache-2.0 e roda localmente.
WHISPER_ID = 'openai/whisper-small'
WHISPER_REV = '973afd24965f72e36ca33b3055d56a652f456b4d'

ROOT.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(ROOT).free < 15 * 1024**3:
    raise RuntimeError('Menos de 15 GiB livres em /content; libere espaço antes de baixar os dois candidatos.')
# Mantém cache de pesos e pip dentro do diretório removível do experimento.
CHILD_ENV = os.environ.copy()
CHILD_ENV['HF_HOME'] = str(ROOT / 'hf_home')
CHILD_ENV['PIP_CACHE_DIR'] = str(ROOT / 'pip_cache')
CHILD_ENV['TORCH_HOME'] = str(ROOT / 'torch_home')
CHILD_ENV['PYTHONFAULTHANDLER'] = '1'
CHILD_ENV['USE_TF'] = '0'
CHILD_ENV['USE_FLAX'] = '0'

def run_streamed(cmd, label, env=None, timeout_seconds=None):
    """Stream child logs into the Colab cell, including setup/preflight failures."""
    print(f'\n[{label}] iniciando; logs do processo aparecem abaixo...', flush=True)
    proc = subprocess.Popen(cmd, env=env or CHILD_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    assert proc.stdout is not None
    timed_out = threading.Event()
    timer = None
    if timeout_seconds is not None:
        def stop_timed_out_process():
            timed_out.set()
            proc.kill()
        timer = threading.Timer(timeout_seconds, stop_timed_out_process)
        timer.daemon = True
        timer.start()
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
        code = proc.wait()
    finally:
        if timer is not None:
            timer.cancel()
    if timed_out.is_set():
        raise TimeoutError(f'{label} excedeu {timeout_seconds}s; processo encerrado para evitar espera indefinida.')
    if code:
        raise RuntimeError(f'{label} encerrou com código {code}; consulte o log do processo imediatamente acima.')

# Confirma GPU antes de baixar vários GB ou instalar pacotes.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Este teste requer GPU CUDA ativa no Colab (preferência: T4). Não baixei modelos. Ative GPU e execute a célula novamente.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM total: %.1f GiB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
print('Torch existente:', torch.__version__, '| CUDA:', torch.version.cuda)

# Upload local privado ao runtime Colab; não usa referência de voz em API hospedada.
if not REF_WAV.exists():
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Execute no Colab ou coloque o WAV original em ' + str(REF_WAV)) from exc
    print('Envie o WAV ORIGINAL limpo da Lia (não o arquivo gerado).')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Envie exatamente um WAV original.')
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith('.wav'):
        raise ValueError('A referência precisa ser WAV para este teste.')
    REF_WAV.write_bytes(uploaded_bytes)

# Confere duração e formato antes de iniciar downloads grandes.
import soundfile as sf
ref_audio, ref_sr = sf.read(str(REF_WAV), dtype='float32', always_2d=True)
ref_seconds = len(ref_audio) / ref_sr
print(f'Referência: {REF_WAV.name}; {ref_sr} Hz; {ref_seconds:.2f} s; {ref_audio.shape[1]} canal(is).')
if ref_seconds < 2.0:
    raise ValueError('Referência muito curta (<2 s). Escolha uma gravação limpa mais longa.')
if ref_seconds < 5.0:
    print('AVISO: referência abaixo de 5 s; a clonagem pode ficar menos estável. Se puder, use 6–15 s de fala limpa da mesma voz.')
# A transcrição é estimada localmente pelo Whisper no venv isolado e mostrada
# para revisão antes do Qwen; nenhum áudio é enviado a uma API de ASR.

# Os Spaces fixam Torch/Torchaudio 2.8.0; para a T4, use os wheels CUDA 12.8.
# Não herde os binários Torch 2.11/CUDA 13 nem a torchvision global do Colab:
# instale Torch/Torchaudio isolados e desative somente a integração opcional de visão.
TORCH_RUNTIME = ROOT / 'torch_runtime'
TORCH_RUNTIME_MARKER = ROOT / '.torch_runtime_ready'
if not TORCH_RUNTIME_MARKER.exists():
    if TORCH_RUNTIME.exists():
        shutil.rmtree(TORCH_RUNTIME)
    TORCH_RUNTIME.mkdir(parents=True, exist_ok=True)
    torch_cmd = [
        sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
        '--no-warn-script-location', '--no-cache-dir', '--only-binary=:all:', '--ignore-installed',
        '--target', str(TORCH_RUNTIME),
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--extra-index-url', 'https://pypi.org/simple',
        'torch==2.8.0+cu128', 'torchaudio==2.8.0+cu128', 'numpy>=2.1,<2.4',
    ]
    subprocess.run(torch_cmd, check=True, env=CHILD_ENV)
    TORCH_RUNTIME_MARKER.write_text('torch 2.8.0+cu128; torchaudio 2.8.0+cu128; numpy 2.1–2.3\n', encoding='utf-8')
else:
    print('Reutilizando pilha Torch isolada em', TORCH_RUNTIME)

# Protobuf 6.x precisa preceder o Protobuf antigo do Colab. O namespace
# `google` pode ser um pacote regular global que esconde a pasta isolada; os
# runners estendem explicitamente google.__path__ antes de importar protobuf.
PROTOBUF_INIT = TORCH_RUNTIME / 'google' / 'protobuf' / '__init__.py'
if not PROTOBUF_INIT.is_file():
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
        '--no-warn-script-location', '--no-cache-dir', '--ignore-installed', '--no-deps',
        '--target', str(TORCH_RUNTIME), 'protobuf>=6.31.1,<7',
    ], check=True, env=CHILD_ENV)
    if not PROTOBUF_INIT.is_file():
        raise RuntimeError('pip não instalou Protobuf dentro de ' + str(TORCH_RUNTIME))
    (ROOT / '.protobuf_runtime_ready').write_text('isolated protobuf>=6.31.1,<7\\n', encoding='utf-8')

probe_env = CHILD_ENV.copy()
probe_env['PYTHONPATH'] = str(TORCH_RUNTIME) + os.pathsep + probe_env.get('PYTHONPATH', '')
probe_env['LIA_TORCH_RUNTIME'] = str(TORCH_RUNTIME)
probe = r"""import os, sys, importlib.metadata, google
from pathlib import Path
local_google = Path(os.environ['LIA_TORCH_RUNTIME']) / 'google'
google.__path__ = [str(local_google), *[p for p in google.__path__ if str(p) != str(local_google)]]
for _name in list(sys.modules):
    if _name == 'google.protobuf' or _name.startswith('google.protobuf.'):
        del sys.modules[_name]
import google.protobuf, torch, torchaudio
print('Isolated Protobuf:', google.protobuf.__version__, google.protobuf.__file__)
print('Isolated Torch/Torchaudio:', torch.__version__, torchaudio.__version__)
print('Triton wheel metadata:', importlib.metadata.version('triton'))
print('CUDA available:', torch.cuda.is_available())
assert tuple(map(int, google.protobuf.__version__.split('.')[:3])) >= (6, 31, 1)
assert Path(google.protobuf.__file__).resolve().is_relative_to(local_google.resolve()), 'Protobuf came from outside the isolated runtime'
assert torch.__version__.startswith('2.8.0+cu128')
assert torchaudio.__version__.startswith('2.8.0+cu128')
assert torch.cuda.is_available(), 'Isolated CUDA stack cannot see the Colab GPU'
print('GPU:', torch.cuda.get_device_name(0))
"""
run_streamed([sys.executable, '-X', 'faulthandler', '-u', '-c', probe], 'Pré-verificação CUDA/Protobuf', env=probe_env)

# Venvs de bibliotecas de aplicação; o Torch compatível fica isolado em TORCH_RUNTIME.
# Nada altera os pacotes globais do kernel.
def make_env(name, packages):
    env_dir = ROOT / ('venv-' + name)
    py = env_dir / 'bin' / 'python'
    if not py.exists() or not (env_dir / 'pyvenv.cfg').exists():
        if env_dir.exists():
            shutil.rmtree(env_dir)
        # Python 3.13 Colab may omit ensurepip; the runtime pip installs into this venv's target dir below.
        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(str(env_dir))
    # Exponha dependências auxiliares do kernel sem alterá-las; os pacotes
    # explícitos do experimento (Torch e protobuf) têm precedência nos runners.
    site_dirs = [p for p in sys.path if p and ('site-packages' in p or 'dist-packages' in p) and Path(p).is_dir()]
    venv_site = next((p for p in (env_dir / 'lib').glob('python*/site-packages')), None)
    if venv_site is None:
        raise RuntimeError('Não encontrei site-packages no ambiente isolado ' + str(env_dir))
    if site_dirs:
        (venv_site / 'colab_kernel_paths.pth').write_text('\n'.join(dict.fromkeys(site_dirs)) + '\n', encoding='utf-8')
    marker = env_dir / '.lia_deps_ready_v2'
    if not marker.exists():
        cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
              '--no-warn-script-location', '--upgrade', '--target', str(venv_site), *packages]
        subprocess.run(cmd, check=True, env=CHILD_ENV)
        marker.write_text('ok\n', encoding='utf-8')
    return py

# ---------- Chatterbox: pack PT-BR dedicado + referência da Lia ----------
chat_py = make_env('chatterbox_ptbr', [
    'numpy>=2.1,<2.4', 'resampy==0.4.3', 'librosa==0.11.0', 's3tokenizer',
    'transformers==4.46.3', 'diffusers==0.29.0', 'omegaconf==2.3.0',
    'resemble-perth==1.0.1', 'silero-vad==5.1.2', 'conformer==0.3.2',
    'safetensors', 'huggingface_hub==0.30.2', 'protobuf>=6.31.1,<7',
])

# Transcreve a referência no próprio Colab, em processo separado e com modelo pinado.
# O resultado aprovado é reutilizado para o mesmo WAV/modelo, sem repetir ASR em cada rodada.
if not REF_TEXT.strip():
    transcript_path=ROOT/'whisper_ref_transcription.json'
    reference_sha256=hashlib.sha256(REF_WAV.read_bytes()).hexdigest()
    whisper_result=None
    if transcript_path.exists():
        try:
            candidate=json.loads(transcript_path.read_text(encoding='utf-8'))
            if (candidate.get('reference_audio_sha256')==reference_sha256
                and candidate.get('model')==WHISPER_ID
                and candidate.get('revision')==WHISPER_REV):
                whisper_result=candidate
        except (OSError, ValueError, json.JSONDecodeError):
            whisper_result=None
    if whisper_result is None:
        whisper_runner = ROOT / 'run_local_whisper.py'
        whisper_runner.write_text(textwrap.dedent(r'''
            import hashlib, json, sys, time
            from pathlib import Path
            root=Path(sys.argv[1]); wav_path=Path(sys.argv[2]); out_path=Path(sys.argv[3])
            model_id=sys.argv[4]; revision=sys.argv[5]
            sys.path.insert(0, str(root / 'torch_runtime'))
            import google
            local_google=str(root / 'torch_runtime' / 'google')
            google.__path__=[local_google, *[p for p in google.__path__ if str(p)!=local_google]]
            for name in list(sys.modules):
                if name=='google.protobuf' or name.startswith('google.protobuf.'):
                    del sys.modules[name]
            import google.protobuf
            assert tuple(map(int, google.protobuf.__version__.split('.')[:3])) >= (6,31,1)
            import librosa, numpy as np, soundfile as sf, torch
            import transformers.utils.import_utils as _hf_import_utils
            _hf_import_utils._torchvision_available = False
            from transformers import WhisperForConditionalGeneration, WhisperProcessor
            audio,sr=sf.read(str(wav_path),dtype='float32',always_2d=True)
            mono=audio.mean(axis=1)
            if sr != 16000:
                mono=librosa.resample(mono,orig_sr=sr,target_sr=16000)
            t0=time.perf_counter()
            processor=WhisperProcessor.from_pretrained(model_id,revision=revision)
            model=WhisperForConditionalGeneration.from_pretrained(
                model_id,revision=revision,torch_dtype=torch.float16,use_safetensors=True).to('cuda:0')
            inputs=processor(mono,sampling_rate=16000,return_tensors='pt',return_attention_mask=True)
            features=inputs.input_features.to(device='cuda:0',dtype=torch.float16)
            attention_mask=inputs.attention_mask.to(device='cuda:0')
            forced=processor.get_decoder_prompt_ids(language='portuguese',task='transcribe')
            with torch.inference_mode():
                ids=model.generate(features,attention_mask=attention_mask,forced_decoder_ids=forced,max_new_tokens=256,do_sample=False)
            transcript=processor.batch_decode(ids,skip_special_tokens=True)[0].strip()
            if not transcript:
                raise RuntimeError('Whisper não encontrou fala reconhecível no WAV de referência.')
            result={'model':model_id,'revision':revision,'language':'Portuguese','transcript':transcript,
                    'reference_audio_sha256':hashlib.sha256(wav_path.read_bytes()).hexdigest(),
                    'audio_seconds':len(audio)/sr,'inference_seconds':time.perf_counter()-t0}
            out_path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
            print(json.dumps(result,ensure_ascii=False),flush=True)
        '''),encoding='utf-8')
        print('Transcrevendo o WAV localmente com Whisper-small; o áudio não é enviado a serviços externos.')
        run_streamed([str(chat_py),'-X','faulthandler','-u',str(whisper_runner),str(ROOT),str(REF_WAV),str(transcript_path),WHISPER_ID,WHISPER_REV],'ASR local da referência',timeout_seconds=180)
        whisper_result=json.loads(transcript_path.read_text(encoding='utf-8'))
    else:
        print('Reutilizando a transcrição Whisper em cache: mesmo SHA-256 do WAV e mesma revisão do modelo.')
    if whisper_result.get('approved_transcript'):
        REF_TEXT=whisper_result['approved_transcript']
        print('Usando a transcrição já revisada; ASR e confirmação manual foram dispensados.')
    else:
        print('Transcrição automática — confira pontuação, palavras e nomes próprios:')
        print(whisper_result['transcript'])
        correction=input('Enter para aceitar; ou cole a transcrição corrigida para usar no Qwen: ').strip()
        REF_TEXT=correction or whisper_result['transcript']
        whisper_result['approved_transcript']=REF_TEXT
        transcript_path.write_text(json.dumps(whisper_result,ensure_ascii=False,indent=2),encoding='utf-8')
else:
    print('Usando REF_TEXT fornecido na configuração; confirme que corresponde ao WAV inteiro.')
print(f'Ref_text final: {len(REF_TEXT)} caracteres; Qwen language={QWEN_LANGUAGE}.')

space_dir = ROOT / 'chatterbox_ptbr_space'
if not space_dir.exists():
    subprocess.run(['git', 'clone', 'https://huggingface.co/spaces/' + SPACE_ID, str(space_dir)], check=True, env=CHILD_ENV)
subprocess.run(['git', '-C', str(space_dir), 'fetch', '--all'], check=True, env=CHILD_ENV)
subprocess.run(['git', '-C', str(space_dir), 'checkout', '--force', SPACE_REV], check=True, env=CHILD_ENV)

chat_runner = ROOT / 'run_chatterbox_ptbr.py'
chat_runner.write_text(textwrap.dedent(r'''
    import hashlib, json, sys, time
    from pathlib import Path
    root = Path(sys.argv[1]); space_dir = Path(sys.argv[2])
    ref_path = Path(sys.argv[3]); texts = json.loads(Path(sys.argv[4]).read_text(encoding='utf-8'))
    ref_sha256=hashlib.sha256(ref_path.read_bytes()).hexdigest()
    # Prefer the locally installed, matched CUDA stack before Colab's global packages.
    sys.path.insert(0, str(root / 'torch_runtime'))
    import google
    _local_google = str(root / 'torch_runtime' / 'google')
    google.__path__ = [_local_google, *[p for p in google.__path__ if str(p) != _local_google]]
    for _name in list(sys.modules):
        if _name == 'google.protobuf' or _name.startswith('google.protobuf.'):
            del sys.modules[_name]
    import google.protobuf
    print('Protobuf do runner:', google.protobuf.__version__, google.protobuf.__file__, flush=True)
    assert tuple(map(int, google.protobuf.__version__.split('.')[:3])) >= (6, 31, 1), 'protobuf isolado não está no sys.path do runner'
    import numpy as np
    import soundfile as sf
    import torch
    from huggingface_hub import snapshot_download
    # TTS is audio/text-only; avoid importing Colab's unrelated global torchvision.
    import transformers.utils.import_utils as _hf_import_utils
    _hf_import_utils._torchvision_available = False
    sys.path.insert(0, str(space_dir))
    from chatterbox.src.chatterbox.tts import ChatterboxTTS

    assets = root / 'chatterbox_ptbr_assets'
    assets.mkdir(parents=True, exist_ok=True)
    base = Path(snapshot_download(
        repo_id='ResembleAI/chatterbox',
        revision='5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18',
        allow_patterns=['ve.pt', 'conds.pt'],
    ))
    ptbr = Path(snapshot_download(
        repo_id='ResembleAI/Chatterbox-Multilingual-pt-br',
        revision='b3952f18bc2eaa72b9bd7c17d2c4653bcad4770d',
        allow_patterns=['t3_pt_br.safetensors', 's3gen_v3.pt', 'grapheme_mtl_merged_expanded_v1.json'],
    ))
    for name, folder in [
        ('ve.pt', base), ('conds.pt', base),
        ('t3_pt_br.safetensors', ptbr), ('s3gen_v3.pt', ptbr),
        ('grapheme_mtl_merged_expanded_v1.json', ptbr),
    ]:
        target = assets / name
        if not target.exists():
            target.symlink_to(folder / name)

    t0 = time.perf_counter()
    model = ChatterboxTTS.from_local(str(assets), device='cuda')
    load_s = time.perf_counter() - t0
    print(f'Chatterbox PT-BR carregado em {load_s:.2f}s; sample rate={model.sr}.')
    model.prepare_conditionals(str(ref_path), exaggeration=0.5)
    rows = []
    for i, text in enumerate(texts, 1):
        torch.manual_seed(1234)
        t0 = time.perf_counter()
        wav = model.generate(text, language_id='pt', exaggeration=0.5, temperature=0.8, cfg_weight=0.5)
        elapsed = time.perf_counter() - t0
        arr = wav.squeeze().detach().float().cpu().numpy() if torch.is_tensor(wav) else np.asarray(wav).squeeze()
        path = root / f'chatterbox_ptbr_{i:02d}.wav'
        sf.write(path, arr, model.sr)
        duration = len(arr) / model.sr
        row = {'model':'Chatterbox-Multilingual-pt-br','text':text,'path':str(path),'duration_s':duration,'generation_s':elapsed,'rtf':elapsed/max(duration,1e-9)}
        rows.append(row)
        print(json.dumps(row, ensure_ascii=False))
    (root / 'chatterbox_ptbr_metrics.json').write_text(json.dumps({'model_repo':'ResembleAI/Chatterbox-Multilingual-pt-br','model_revision':'b3952f18bc2eaa72b9bd7c17d2c4653bcad4770d','base_repo':'ResembleAI/chatterbox','base_revision':'5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18','space_revision':'9e515821e826e207cd617a0fdd0223899ed108ea','reference_audio_sha256':ref_sha256,'load_seconds':load_s,'results':rows},ensure_ascii=False,indent=2),encoding='utf-8')
'''), encoding='utf-8')

texts_json = ROOT / 'texts.json'
texts_json.write_text(json.dumps(TEXTS, ensure_ascii=False), encoding='utf-8')
print('\nExecutando Chatterbox PT-BR (referência permanece no Colab)...')
chat_metrics_path = ROOT / 'chatterbox_ptbr_metrics.json'
chat_reference_sha256=hashlib.sha256(REF_WAV.read_bytes()).hexdigest()
reuse_chat = False
if chat_metrics_path.is_file():
    try:
        previous = json.loads(chat_metrics_path.read_text(encoding='utf-8'))
        reuse_chat = (
            previous.get('space_revision') == SPACE_REV
            and previous.get('reference_audio_sha256') == chat_reference_sha256
            and [row.get('text') for row in previous.get('results', [])] == TEXTS
            and all(Path(row.get('path', '')).is_file() for row in previous.get('results', []))
        )
    except Exception:
        reuse_chat = False
if reuse_chat:
    print('Reutilizando as três amostras Chatterbox já concluídas em', ROOT)
else:
    run_streamed([str(chat_py), '-X', 'faulthandler', '-u', str(chat_runner), str(ROOT), str(space_dir), str(REF_WAV), str(texts_json)], 'Chatterbox PT-BR')

# A/B controlado de prosódia Chatterbox: mesmo texto, referência, seed e temperature.
# Só mudam exaggeration/cfg_weight; valores exploratórios baseados nas dicas upstream.
expression_metrics_path=ROOT/'chatterbox_expression_metrics.json'
reference_sha256=hashlib.sha256(REF_WAV.read_bytes()).hexdigest()
reuse_expression=False
if expression_metrics_path.is_file():
    try:
        m=json.loads(expression_metrics_path.read_text(encoding='utf-8'))
        reuse_expression=(m.get('model_revision')==CHATTERBOX_PTBR_REV
            and m.get('reference_audio_sha256')==reference_sha256
            and m.get('text')==EXPRESSION_TEST_TEXT
            and [(r.get('label'),r.get('exaggeration'),r.get('cfg_weight'),r.get('temperature'),r.get('seed')) for r in m.get('results',[])]==[('baseline',0.5,0.5,0.8,1234),('exaggeration_only',0.7,0.5,0.8,1234),('cfg_only',0.5,0.3,0.8,1234),('recommended_combo',0.7,0.3,0.8,1234)]
            and all(Path(r.get('path','')).is_file() for r in m['results']))
    except Exception:
        reuse_expression=False
if not reuse_expression:
    expression_runner=ROOT/'run_chatterbox_expression.py'
    expression_runner.write_text(textwrap.dedent(r'''
        import hashlib, json, sys, time
        from pathlib import Path
        root=Path(sys.argv[1]); space_dir=Path(sys.argv[2]); ref_path=Path(sys.argv[3]); text=sys.argv[4]
        sys.path.insert(0,str(root/'torch_runtime'))
        import google
        local_google=str(root/'torch_runtime'/'google')
        google.__path__=[local_google,*[p for p in google.__path__ if str(p)!=local_google]]
        for name in list(sys.modules):
            if name=='google.protobuf' or name.startswith('google.protobuf.'):
                del sys.modules[name]
        import google.protobuf, numpy as np, soundfile as sf, torch
        import transformers.utils.import_utils as _hf_import_utils
        _hf_import_utils._torchvision_available=False
        from huggingface_hub import snapshot_download
        sys.path.insert(0,str(space_dir))
        from chatterbox.src.chatterbox.tts import ChatterboxTTS
        root_assets=root/'chatterbox_ptbr_assets'; root_assets.mkdir(parents=True,exist_ok=True)
        base=Path(snapshot_download(repo_id='ResembleAI/chatterbox',revision='5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18',allow_patterns=['ve.pt','conds.pt']))
        ptbr=Path(snapshot_download(repo_id='ResembleAI/Chatterbox-Multilingual-pt-br',revision='b3952f18bc2eaa72b9bd7c17d2c4653bcad4770d',allow_patterns=['t3_pt_br.safetensors','s3gen_v3.pt','grapheme_mtl_merged_expanded_v1.json']))
        for name,folder in [('ve.pt',base),('conds.pt',base),('t3_pt_br.safetensors',ptbr),('s3gen_v3.pt',ptbr),('grapheme_mtl_merged_expanded_v1.json',ptbr)]:
            target=root_assets/name
            if not target.exists(): target.symlink_to(folder/name)
        t0=time.perf_counter(); model=ChatterboxTTS.from_local(str(root_assets),device='cuda'); load_s=time.perf_counter()-t0
        model.prepare_conditionals(str(ref_path),exaggeration=0.5)
        configs=[('baseline',0.5,0.5),('exaggeration_only',0.7,0.5),('cfg_only',0.5,0.3),('recommended_combo',0.7,0.3)]
        results=[]
        for label,exaggeration,cfg_weight in configs:
            torch.manual_seed(1234); t0=time.perf_counter()
            wav=model.generate(text,language_id='pt',exaggeration=exaggeration,temperature=0.8,cfg_weight=cfg_weight)
            elapsed=time.perf_counter()-t0
            arr=wav.squeeze().detach().float().cpu().numpy() if torch.is_tensor(wav) else np.asarray(wav).squeeze()
            duration=len(arr)/model.sr; path=root/f'chatterbox_expression_{label}.wav'; sf.write(path,arr,model.sr)
            row={'label':label,'model':'Chatterbox-Multilingual-pt-br','text':text,'path':str(path),'exaggeration':exaggeration,'cfg_weight':cfg_weight,'temperature':0.8,'seed':1234,'duration_s':duration,'generation_s':elapsed,'rtf':elapsed/max(duration,1e-9)}
            results.append(row); print(json.dumps(row,ensure_ascii=False),flush=True)
        metadata={'model_revision':'b3952f18bc2eaa72b9bd7c17d2c4653bcad4770d','reference_audio_sha256':hashlib.sha256(ref_path.read_bytes()).hexdigest(),'load_seconds':load_s,'text':text,'results':results}
        (root/'chatterbox_expression_metrics.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
    '''),encoding='utf-8')
    run_streamed([str(chat_py),'-X','faulthandler','-u',str(expression_runner),str(ROOT),str(space_dir),str(REF_WAV),EXPRESSION_TEST_TEXT],'Chatterbox — A/B de expressividade',timeout_seconds=180)
else:
    print('Reutilizando o A/B de expressividade Chatterbox para o mesmo WAV/texto.')

# ---------- Qwen3-TTS: fine-tunable Base 0.6B, português genérico ----------
qwen_py = make_env('qwen3_tts', [
    '--no-deps', 'qwen-tts==0.1.1',
])
qwen_site = next((p for p in (qwen_py.parent.parent / 'lib').glob('python*/site-packages')), None)
if qwen_site is None:
    raise RuntimeError('Não encontrei site-packages no ambiente isolado Qwen.')
qwen_deps_marker=qwen_py.parent/'.lia_qwen_deps_ready_v1'
if not qwen_deps_marker.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-warn-script-location',
                    '--upgrade', '--target', str(qwen_site), 'transformers==4.57.3', 'accelerate==1.12.0', 'librosa', 'soundfile', 'sox', 'onnxruntime', 'einops', 'huggingface_hub==0.36.2', 'protobuf>=6.31.1,<7'], check=True, env=CHILD_ENV)
    qwen_deps_marker.write_text('qwen runner dependencies installed in isolated venv\n',encoding='utf-8')
else:
    print('Reutilizando dependências Qwen isoladas já instaladas.')

qwen_runner = ROOT / 'run_qwen_pt.py'
qwen_runner.write_text(textwrap.dedent(r'''
    import hashlib, json, sys, time
    from pathlib import Path
    root=Path(sys.argv[1]); ref_path=Path(sys.argv[2]); ref_text=sys.argv[3]
    texts=json.loads(Path(sys.argv[4]).read_text(encoding='utf-8'))
    max_new_tokens=int(sys.argv[5]); max_audio_seconds=float(sys.argv[6]); language=sys.argv[7]
    expression_text=sys.argv[8]
    # Use the same isolated Torch/CUDA stack as the Chatterbox run.
    sys.path.insert(0, str(root / 'torch_runtime'))
    import google
    _local_google = str(root / 'torch_runtime' / 'google')
    google.__path__ = [_local_google, *[p for p in google.__path__ if str(p) != _local_google]]
    for _name in list(sys.modules):
        if _name == 'google.protobuf' or _name.startswith('google.protobuf.'):
            del sys.modules[_name]
    import google.protobuf
    print('Protobuf do runner:', google.protobuf.__version__, google.protobuf.__file__, flush=True)
    assert tuple(map(int, google.protobuf.__version__.split('.')[:3])) >= (6, 31, 1), 'protobuf isolado não está no sys.path do runner'
    import soundfile as sf
    import numpy as np
    import torch
    from huggingface_hub import snapshot_download
    # Qwen TTS also does not need vision; keep the kernel's optional torchvision out.
    import transformers.utils.import_utils as _hf_import_utils
    _hf_import_utils._torchvision_available = False
    from qwen_tts import Qwen3TTSModel
    ref_info=sf.info(str(ref_path))
    ref_sha256=hashlib.sha256(ref_path.read_bytes()).hexdigest()
    model_dir=snapshot_download(
        repo_id='Qwen/Qwen3-TTS-12Hz-0.6B-Base',
        revision='5d83992436eae1d760afd27aff78a71d676296fc',
    )
    t0=time.perf_counter()
    model=Qwen3TTSModel.from_pretrained(
        model_dir, device_map='cuda:0', dtype=torch.float32,
        attn_implementation='sdpa',
    )
    load_s=time.perf_counter()-t0
    print(f'Qwen3-TTS Base 0.6B carregado em {load_s:.2f}s (FP32/SDPA);')
    rows=[]
    for i,text in enumerate(texts,1):
        torch.manual_seed(1234)
        t0=time.perf_counter()
        wavs,sr=model.generate_voice_clone(
            text=text, language=language, ref_audio=str(ref_path), ref_text=ref_text,
            max_new_tokens=max_new_tokens,
        )
        elapsed=time.perf_counter()-t0
        arr=wavs[0]
        duration=len(arr)/sr
        if not np.isfinite(arr).all():
            raise RuntimeError(f'Qwen frase {i}: áudio contém valores não finitos; não será salvo/reproduzido.')
        if duration > max_audio_seconds:
            raise RuntimeError(
                f'Qwen frase {i}: áudio patológico de {duration:.2f}s excede o limite '
                f'de {max_audio_seconds:.0f}s; não será salvo/reproduzido. '
                f'O limite de geração foi {max_new_tokens} tokens.'
            )
        path=root/f'qwen3_tts_pt_{i:02d}.wav'
        sf.write(path,arr,sr)
        row={'model':'Qwen3-TTS-12Hz-0.6B-Base','text':text,'path':str(path),'duration_s':duration,'generation_s':elapsed,'rtf':elapsed/max(duration,1e-9)}
        rows.append(row)
        print(json.dumps(row,ensure_ascii=False))
    torch.manual_seed(1234)
    t0=time.perf_counter()
    expression_wavs,expression_sr=model.generate_voice_clone(
        text=expression_text,language=language,ref_audio=str(ref_path),ref_text=ref_text,
        max_new_tokens=max_new_tokens,
    )
    expression_elapsed=time.perf_counter()-t0
    expression_arr=expression_wavs[0]
    expression_duration=len(expression_arr)/expression_sr
    if not np.isfinite(expression_arr).all():
        raise RuntimeError('Qwen expressividade: áudio contém valores não finitos; não será salvo/reproduzido.')
    if expression_duration > max_audio_seconds:
        raise RuntimeError(f'Qwen expressividade: áudio de {expression_duration:.2f}s excede {max_audio_seconds:.0f}s; não será salvo/reproduzido.')
    expression_path=root/'qwen3_tts_expression.wav'
    sf.write(expression_path,expression_arr,expression_sr)
    expression_row={'model':'Qwen3-TTS-12Hz-0.6B-Base','label':'semantic_emotion_default_sampling','text':expression_text,'path':str(expression_path),'duration_s':expression_duration,'generation_s':expression_elapsed,'rtf':expression_elapsed/max(expression_duration,1e-9)}
    rows.append(expression_row)
    print(json.dumps(expression_row,ensure_ascii=False))
    (root/'qwen3_tts_metrics.json').write_text(json.dumps({'model_repo':'Qwen/Qwen3-TTS-12Hz-0.6B-Base','model_revision':'5d83992436eae1d760afd27aff78a71d676296fc','package':'qwen-tts==0.1.1','dtype':'float32','language':language,'max_new_tokens':max_new_tokens,'max_audio_seconds':max_audio_seconds,'reference_audio_sha256':ref_sha256,'reference_audio_seconds':ref_info.duration,'reference_audio_sample_rate':ref_info.samplerate,'reference_transcript':ref_text,'load_seconds':load_s,'results':rows},ensure_ascii=False,indent=2),encoding='utf-8')
'''), encoding='utf-8')

qwen_metrics_path=ROOT/'qwen3_tts_metrics.json'
reuse_qwen=False
if qwen_metrics_path.is_file():
    try:
        previous_qwen=json.loads(qwen_metrics_path.read_text(encoding='utf-8'))
        expected_texts=TEXTS+[EXPRESSION_TEST_TEXT]
        reuse_qwen=(previous_qwen.get('model_revision')==QWEN_REV
            and previous_qwen.get('reference_audio_sha256')==reference_sha256
            and previous_qwen.get('reference_transcript')==REF_TEXT
            and previous_qwen.get('language')==QWEN_LANGUAGE
            and previous_qwen.get('max_new_tokens')==QWEN_MAX_NEW_TOKENS
            and previous_qwen.get('max_audio_seconds')==QWEN_MAX_AUDIO_SECONDS
            and [r.get('text') for r in previous_qwen.get('results',[])]==expected_texts
            and all(Path(r.get('path','')).is_file() for r in previous_qwen.get('results',[])))
    except Exception:
        reuse_qwen=False
if reuse_qwen:
    print('Reutilizando Qwen: mesmo hash da referência, transcript, modelo e parâmetros.')
else:
    print('\nExecutando Qwen3-TTS Base 0.6B (ref_text deve corresponder literalmente ao WAV)...')
    run_streamed([str(qwen_py), '-X', 'faulthandler', '-u', str(qwen_runner), str(ROOT), str(REF_WAV), REF_TEXT, str(texts_json), str(QWEN_MAX_NEW_TOKENS), str(QWEN_MAX_AUDIO_SECONDS), QWEN_LANGUAGE, EXPRESSION_TEST_TEXT], 'Qwen3-TTS', timeout_seconds=180)

# Toca todas as amostras no notebook, intercaladas por frase para comparação cega.
from IPython.display import Audio, display, Markdown
print('\nConcluído. Arquivos e métricas em:', ROOT)
for i, text in enumerate(TEXTS, 1):
    display(Markdown(f'### Frase {i}: {text}'))
    for label, filename in [
        ('Chatterbox PT-BR', f'chatterbox_ptbr_{i:02d}.wav'),
        ('Qwen3-TTS Base', f'qwen3_tts_pt_{i:02d}.wav'),
    ]:
        print(label)
        display(Audio(filename=str(ROOT / filename)))

display(Markdown(f'### Expressividade (mesmo texto para todos): {EXPRESSION_TEST_TEXT}'))
for label, filename in [
    ('Chatterbox baseline (exaggeration 0.5 / CFG 0.5)', 'chatterbox_expression_baseline.wav'),
    ('Chatterbox exaggeration only (0.7 / 0.5)', 'chatterbox_expression_exaggeration_only.wav'),
    ('Chatterbox CFG only (0.5 / 0.3)', 'chatterbox_expression_cfg_only.wav'),
    ('Chatterbox combo expressivo (0.7 / 0.3)', 'chatterbox_expression_recommended_combo.wav'),
    ('Qwen Base (amostragem padrão; sem controle instruct)', 'qwen3_tts_expression.wav'),
]:
    print(label)
    display(Audio(filename=str(ROOT / filename)))

print('Latência de síntese (não é streaming/TTFA) e RTF:')
for fn in ['chatterbox_ptbr_metrics.json','qwen3_tts_metrics.json']:
    data=json.loads((ROOT/fn).read_text(encoding='utf-8'))
    print(fn, '| dtype=%s | carga=%.2fs' % (data.get('dtype', 'default'), data['load_seconds']))
    for r in data['results']:
        print(r['model'], '| geração=%.2fs | áudio=%.2fs | RTF=%.3f |' % (r['generation_s'],r['duration_s'],r['rtf']), r['text'])
expr_data=json.loads((ROOT/'chatterbox_expression_metrics.json').read_text(encoding='utf-8'))
print('A/B Chatterbox (exaggeration / CFG / RTF):')
for r in expr_data['results']:
    print(r['label'], '| %.1f / %.1f | RTF=%.3f | geração=%.2fs | áudio=%.2fs' % (r['exaggeration'],r['cfg_weight'],r['rtf'],r['generation_s'],r['duration_s']))
